# ACS FLT Availability Check

Queries MAST for FLT products from proposal 14706 (ACS/WFC) and attempts to download them.
If no files are downloaded, it confirms all ACS exposures were delivered as FLC only
(i.e. CalACS applied CTE correction to every exposure).

In [ ]:
import shutil
import yaml
from pathlib import Path
from astropy.table import Table
from astroquery.mast import Observations

cfg = yaml.safe_load(open('../config/wfc3_ir_drizzle_params.yaml'))['acs_download']
RAW_DIR = Path('../data_acs/raw')

# Query MAST — same criteria as the main ACS download
obs_table = Observations.query_criteria(
    proposal_id=cfg['proposal_id'],
    instrument_name=cfg['instrument_name'],
    project=cfg['project']
)
products = Observations.get_product_list(obs_table)

# Filter to FLT only
flt_mask = products['productSubGroupDescription'] == 'FLT'
flt_products = products[flt_mask]

print(f'Total ACS products available: {len(products)}')
print(f'FLT products found:           {len(flt_products)}')
if len(flt_products) == 0:
    print('\nNo FLT files exist for this proposal — all exposures were delivered as FLC.')
else:
    print('\nFLT files found:')
    display(Table(flt_products[['obs_id', 'productFilename', 'productSubGroupDescription']]))

In [ ]:
# Only runs if FLT files were found above — downloads them into data_acs/raw/
if len(flt_products) == 0:
    print('No FLT files to download.')
else:
    Observations.download_products(
        flt_products,
        productSubGroupDescription='FLT',
        project=cfg['data_type'],
        cache=True
    )

    # Move downloaded files into data_acs/raw/<TARGNAME>/
    import glob
    from astropy.io import fits

    moved = 0
    for flt in glob.glob('./mastDownload/HST/*/*flt.fits'):
        flt_path = Path(flt)
        target = fits.getval(str(flt_path), 'TARGNAME', ext=0)
        dest_dir = RAW_DIR / target
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest = dest_dir / flt_path.name
        shutil.move(str(flt_path), str(dest))
        print(f'Moved {flt_path.name} → {dest}')
        moved += 1

    if moved:
        shutil.rmtree('./mastDownload/')
    print(f'\nDone — {moved} FLT file(s) moved to data_acs/raw/')